In [ ]:
# from databricks.connect import DatabricksSession
# import os
# import config

# os.environ["DATABRICKS_HOST"] = config.DATABRICKS_HOST
# os.environ["DATABRICKS_TOKEN"] = config.DATABRICKS_TOKEN
# os.environ["DATABRICKS_CLUSTER_ID"] = config.DATABRICKS_CLUSTER_ID
# os.environ["DATABRICKS_WORKSPACE_ID"] = config.DATABRICKS_WORKSPACE_ID

# spark = DatabricksSession.builder.getOrCreate()

In [ ]:
from pyspark.sql import SparkSession
import config

DATABRICKS_HOST = config.DATABRICKS_HOST
DATABRICKS_TOKEN = config.DATABRICKS_TOKEN
DATABRICKS_CLUSTER_ID = config.DATABRICKS_CLUSTER_ID

# Spark Connect URL format
remote_uri = f"sc://{DATABRICKS_HOST.replace("https://", "")}:443/;token={DATABRICKS_TOKEN};x-databricks-cluster-id={DATABRICKS_CLUSTER_ID}"

# Create SparkSession using Spark Connect
spark = SparkSession.builder.remote(remote_uri).config("spark.connect.session.connectML.enabled", "true").getOrCreate()

In [2]:
# Catalog name where the model artifacts will be stored in Unity Catalog
CATALOG_NAME = "alexander_booth" 
SCHEMA_NAME = "default" # Schema (database) name within the catalog to organize model artifacts
TABLE_NAME = "breast_cancer_training_data"

In [3]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

# Load as pandas DataFrame
data = load_breast_cancer(as_frame=True)
df_pd = pd.concat([data.data.rename(columns=lambda x: x.replace(' ', '_')), data.target.rename("label")], axis=1)

# Convert to Spark DataFrame
df = spark.createDataFrame(df_pd)
df.write.mode("overwrite").saveAsTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}")

In [4]:
df.show(5)

+-----------+------------+--------------+---------+---------------+----------------+--------------+-------------------+-------------+----------------------+------------+-------------+---------------+----------+----------------+-----------------+---------------+--------------------+--------------+-----------------------+------------+-------------+---------------+----------+----------------+-----------------+---------------+--------------------+--------------+-----------------------+-----+
|mean_radius|mean_texture|mean_perimeter|mean_area|mean_smoothness|mean_compactness|mean_concavity|mean_concave_points|mean_symmetry|mean_fractal_dimension|radius_error|texture_error|perimeter_error|area_error|smoothness_error|compactness_error|concavity_error|concave_points_error|symmetry_error|fractal_dimension_error|worst_radius|worst_texture|worst_perimeter|worst_area|worst_smoothness|worst_compactness|worst_concavity|worst_concave_points|worst_symmetry|worst_fractal_dimension|label|
+-----------+-